# Xem câu hỏi và câu trả lời của các mô hình

Đọc manifest (segment), câu trả lời của mô hình (`out/*.jsonl` do `tools/ask_models.py` ghi), nhãn người (nếu có) và bộ câu hỏi `data/config.json`.
Gồm hai phần: (1) nghe và kiểm tra nhãn LLM gán cho từng file lúc `collect.py`; (2) câu trả lời của mô hình cho từng segment. Thư mục lấy từ `paths` trong `config/`.

In [ ]:
import glob, json, os, sys, collections
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "tools"))
from ask_models import load_jsonl, questions
import conf
C = conf.load([])          # thư mục lấy từ paths trong config/ (vd. conf.load(["paths.answers=out2"]))
from audit_questions import majority

MANIFESTS = os.path.join(conf.path(C, "manifests"), "*.jsonl")
ANSWERS   = sorted(glob.glob(os.path.join(conf.path(C, "answers"), "*.jsonl")))   # mỗi file một mô hình
HUMANS    = os.path.join(ROOT, "humans.jsonl")                         # bỏ qua nếu chưa có
TAG       = "turntaking"

# Bộ lọc cho phần xem từng segment (dùng khi không có ipywidgets)
FILTER_EVENT    = None      # vd. "MOT"
FILTER_BUCKET   = None      # vd. "interview"
ONLY_DISAGREE   = True      # chỉ các mẫu mô hình bất đồng
FOCUS_QUESTION  = "tcontent.event"
MAX_SHOW        = 10

In [ ]:
import pandas as pd
segs = {r["event_id"]: r for p in sorted(glob.glob(MANIFESTS)) for r in load_jsonl(p)}
QS = questions(TAG)
QTITLE = {k: q["title"] for k, q in QS}
QLABEL = {k: dict(zip(q["codes"], [o[0] for o in q["opts"]])) for k, q in QS}
models = {os.path.splitext(os.path.basename(p))[0]: {r["sample_id"]: r for r in load_jsonl(p)} for p in ANSWERS}
hum = collections.defaultdict(lambda: collections.defaultdict(list))
if os.path.exists(HUMANS):
    for r in load_jsonl(HUMANS):
        for k, v in r["answers"].items():
            hum[r["sample_id"]][k].append(v)
human = {sid: {k: majority([x for x in v if x != "UNK"]) or (v[0] if len(set(v)) == 1 else None) for k, v in d.items()} for sid, d in hum.items()}
SRC = dict(models, **({"người": {sid: {"answers": a} for sid, a in human.items()}} if human else {}))

rows = []
for name, m in SRC.items():
    for sid, r in m.items():
        for k, v in r["answers"].items():
            rows.append({"source": name, "sample_id": sid, "q": k, "a": v})
A = pd.DataFrame(rows, columns=["source", "sample_id", "q", "a"])
HAS_ANS = not A.empty
if not HAS_ANS:
    print(f"Chưa có câu trả lời mức segment trong {conf.path(C, 'answers')} (chạy tools/ask_models.py). Các phần dùng câu trả lời sẽ bỏ qua.")
print(f"{len(segs)} segment, {len(QS)} câu hỏi, nguồn trả lời: {list(SRC)}")
pd.DataFrame([{"mô hình": n, "số mẫu": len(m), "lỗi parse %": 100 * sum(r.get("parse_error", False) for r in m.values()) / max(len(m), 1),
               "UNK %": 100 * (A[(A.source == n)].a == "UNK").mean()} for n, m in models.items()]).round(1)

## Kiểm tra nhãn LLM gán cho từng file (lúc `collect.py`)

Nhãn mức file (genre, accent, channel, noise_type, distance) quyết định file vào bucket nào. Đọc từ `manifests/*.jsonl` (`file_labels`, file đã LẤY) và `<paths.raw>/classified.jsonl` (mọi file, kể cả BỎ, kèm lý do).
Nghe 3 đoạn trích, sửa nhãn nếu sai rồi bấm **Lưu**. Kết quả ghi vào `<paths.raw>/review_file_labels.jsonl`, cell sau tính LLM đúng bao nhiêu %.

In [ ]:
import subprocess
from ask_models import FILE_QS
RAW = conf.path(C, "raw")
REVIEW = os.path.join(RAW, "review_file_labels.jsonl")   # cạnh classified.jsonl, không lẫn với câu trả lời mô hình
FIELD_Q = {k.split(".", 1)[1]: q for k, q in questions(TAG, FILE_QS)}      # genre -> câu hỏi (codes, opts)
EXCERPTS, EXCERPT_SEC = 3, 20

files = {}
cls_path = os.path.join(RAW, "classified.jsonl")
for r in (load_jsonl(cls_path) if os.path.exists(cls_path) else []):
    files[r["id"]] = {"id": r["id"], "src": r["src"], "labels": r["labels"], "decision": r.get("decision", "LẤY" if r.get("used") else "BỎ"),
                      "reason": r.get("reason", ""), "raw": r.get("raw")}
for s in segs.values():                     # file chỉ có trong manifest (vd. collect bản cũ)
    f = files.setdefault(s["conversation_id"], {"id": s["conversation_id"], "src": s.get("source_path"), "labels": s.get("file_labels") or {},
                                                "decision": "LẤY", "reason": ", ".join(s.get("bucket") or []), "raw": None})
    f.setdefault("segments", []).append(s["event_id"])
reviewed = {r["id"]: r for r in load_jsonl(REVIEW)} if os.path.exists(REVIEW) else {}

def file_audio(f):
    """wav đã chuẩn hoá nếu còn (file LẤY), không thì đọc thẳng file nguồn."""
    w = os.path.join(RAW, "files", f["id"] + ".wav")
    return w if os.path.exists(w) else f["src"]

def excerpt_paths(f):
    """EXCERPTS đoạn EXCERPT_SEC giây rải đều trong file, cắt bằng ffmpeg vào <raw>/_work/nb/."""
    src = file_audio(f)
    if not src or not os.path.exists(src):
        return []
    dur = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", src],
                               capture_output=True, text=True, stdin=subprocess.DEVNULL).stdout.strip() or 0)
    d = os.path.join(RAW, "_work", "nb"); os.makedirs(d, exist_ok=True)
    out = []
    for k in range(EXCERPTS):
        t = max(0.0, dur * (k + 0.5) / EXCERPTS - EXCERPT_SEC / 2)
        p = os.path.join(d, f"{f['id']}_{k}.wav")
        if not os.path.exists(p):
            subprocess.run(["ffmpeg", "-nostdin", "-y", "-loglevel", "error", "-ss", str(t), "-t", str(EXCERPT_SEC), "-i", src, "-ac", "1", "-ar", "16000", p], check=True)
        out.append((t, p))
    return out

def transcript_head(f, n=12):
    p = os.path.join(RAW, "files", f["id"] + "_transcript.txt")
    return open(p, encoding="utf8").read().splitlines()[:n] if os.path.exists(p) else []

F = pd.DataFrame([{"id": f["id"], "file": os.path.basename(f["src"] or ""), "quyết định": f["decision"], "lý do": f["reason"],
                   **{k: f["labels"].get(k) for k in list(FIELD_Q) + ["speakers"]}, "đã kiểm tra": f["id"] in reviewed}
                  for f in files.values()])
print(f"{len(F)} file; {F['đã kiểm tra'].sum() if len(F) else 0} đã kiểm tra")
F.head(30)

In [ ]:
from IPython.display import Audio, HTML, display
try:
    import ipywidgets as w
except ImportError:
    w = None

def show_file(f):
    display(HTML(f"<h4>{os.path.basename(f['src'] or f['id'])} · <b>{f['decision']}</b> · {f['reason']}</h4>"
                 f"<p>LLM gán: <b>{' · '.join(f'{k}={v}' for k, v in f['labels'].items())}</b></p>"))
    if f.get("raw"):
        display(HTML(f"<details><summary>Câu trả lời thô của mô hình (lỗi parse)</summary><pre style='white-space:pre-wrap'>{f['raw']}</pre></details>"))
    for t, p in excerpt_paths(f):
        display(HTML(f"<small>đoạn từ giây {t:.0f}</small>")); display(Audio(p))
    tr = transcript_head(f)
    if tr:
        display(HTML("<pre style='white-space:pre-wrap;font-size:12px'>" + "\n".join(tr) + "</pre>"))

def save_review(f, human, reviewer):
    os.makedirs(os.path.dirname(REVIEW), exist_ok=True)
    rec = {"id": f["id"], "src": f["src"], "reviewer": reviewer, "llm": {k: f["labels"].get(k) for k in FIELD_Q},
           "human": human, "at": pd.Timestamp.now().isoformat(timespec="seconds")}
    with open(REVIEW, "a", encoding="utf8") as fh:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    reviewed[f["id"]] = rec

if w is None or not files:
    print("Cần ipywidgets để duyệt bằng giao diện." if files else "Chưa có file nào: chạy tools/collect.py trước.")
    for f in list(files.values())[:3]:
        show_file(f)
else:
    dec_w = w.Dropdown(options=["tất cả", "LẤY", "BỎ"], description="quyết định")
    todo_w = w.Checkbox(value=True, description="chỉ file chưa kiểm tra")
    who_w = w.Text(description="người kiểm", value=os.environ.get("USER", ""))
    sel = w.Dropdown(description="file", layout=w.Layout(width="60%"))
    fields = {k: w.Dropdown(description=k, options=[("(trống)", None)] + [(f"{c} – {o[0]}", c) for c, o in zip(q["codes"], q["opts"])],
                            layout=w.Layout(width="60%")) for k, q in FIELD_Q.items()}
    ok_b, save_b = w.Button(description="LLM đúng hết", button_style="success"), w.Button(description="Lưu nhãn đã sửa", button_style="primary")
    out, msg = w.Output(), w.Label()

    def ids():
        return [i for i, f in files.items() if (dec_w.value == "tất cả" or f["decision"] == dec_w.value) and not (todo_w.value and i in reviewed)]

    def refresh(*_):
        sel.options = [(f"{os.path.basename(files[i]['src'] or i)} [{files[i]['decision']}]", i) for i in ids()] or [("(hết file)", None)]

    def render(*_):
        out.clear_output()
        f = files.get(sel.value)
        if not f:
            return
        for k, d in fields.items():
            d.value = f["labels"].get(k) if f["labels"].get(k) in FIELD_Q[k]["codes"] else None
        with out:
            show_file(f)

    def next_file():
        i = list(sel.options).index(next(o for o in sel.options if o[1] == sel.value)) if sel.value else 0
        refresh()
        if sel.options and sel.options[0][1]:
            sel.value = sel.options[min(i, len(sel.options) - 1)][1]

    def on_save(btn):
        f = files.get(sel.value)
        if not f:
            return
        human = {k: f["labels"].get(k) for k in FIELD_Q} if btn is ok_b else {k: d.value for k, d in fields.items()}
        save_review(f, human, who_w.value)
        diff = [k for k in FIELD_Q if human.get(k) != f["labels"].get(k)]
        msg.value = f"Đã lưu {os.path.basename(f['src'] or f['id'])}: " + (f"sửa {', '.join(diff)}" if diff else "LLM đúng hết")
        next_file()

    for x in (dec_w, todo_w):
        x.observe(refresh, "value")
    sel.observe(render, "value")
    ok_b.on_click(on_save); save_b.on_click(on_save)
    refresh()
    display(w.HBox([dec_w, todo_w, who_w]), sel, w.VBox(list(fields.values())), w.HBox([ok_b, save_b]), msg, out)

### LLM gán đúng bao nhiêu % (từ các file đã kiểm tra)

In [ ]:
R = load_jsonl(REVIEW) if os.path.exists(REVIEW) else []
R = list({r["id"]: r for r in R}.values())          # mỗi file lấy lần kiểm tra mới nhất
if not R:
    print("Chưa kiểm tra file nào.")
else:
    acc = pd.DataFrame([{"nhãn": k, "số file": sum(r["human"].get(k) is not None for r in R),
                         "LLM đúng %": round(100 * sum(r["human"].get(k) == r["llm"].get(k) for r in R if r["human"].get(k) is not None)
                                             / max(1, sum(r["human"].get(k) is not None for r in R)))} for k in FIELD_Q])
    display(acc.sort_values("LLM đúng %"))
    for k in ("genre", "channel"):
        pairs = [(r["human"][k], r["llm"].get(k)) for r in R if r["human"].get(k)]
        if pairs:
            display(HTML(f"<b>{k}</b>: hàng = người, cột = LLM"))
            display(pd.crosstab(pd.Series([a for a, _ in pairs], name="người"), pd.Series([b for _, b in pairs], name="LLM")))

## Phân bố đáp án theo từng câu hỏi

In [ ]:
if not HAS_ANS:
    print("Bỏ qua: chưa có câu trả lời mức segment.")
else:
    import matplotlib.pyplot as plt
    qs = [k for k, _ in QS if k in set(A.q)]
    cols = 3
    fig, axes = plt.subplots((len(qs) + cols - 1) // cols, cols, figsize=(16, 3.2 * ((len(qs) + cols - 1) // cols)), squeeze=False)
    for ax, k in zip(axes.flat, qs):
        t = A[A.q == k].groupby(["a", "source"]).size().unstack(fill_value=0)
        t = (t / t.sum()).reindex([c for c in QLABEL[k] if c in t.index])
        t.plot.bar(ax=ax, width=0.8, legend=ax is axes.flat[0])
        ax.set_title(k, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("tỉ lệ"); ax.tick_params(axis="x", rotation=30, labelsize=8)
    for ax in list(axes.flat)[len(qs):]:
        ax.axis("off")
    plt.tight_layout()

## Đồng thuận giữa các nguồn theo câu hỏi

Sắp từ thấp đến cao: câu ở đầu bảng là câu máy trả lời không ổn định (hoặc lệch người).

In [ ]:
if not HAS_ANS:
    print("Bỏ qua: chưa có câu trả lời mức segment.")
else:
    W = A.pivot_table(index=["sample_id", "q"], columns="source", values="a", aggfunc="first")
    names = list(W.columns)
    agree = {}
    for i, x in enumerate(names):
        for y in names[i + 1:]:
            sub = W[[x, y]].dropna()
            agree[f"{x} ~ {y}"] = (sub[x] == sub[y]).groupby(level="q").mean() * 100
    pd.DataFrame(agree).round(0).sort_values(list(agree)[0] if agree else [], ascending=True) if agree else "cần ít nhất 2 nguồn trả lời"

## Ma trận nhầm lẫn cho một câu

So mô hình với người. Nếu chưa có nhãn người và câu là `tcontent.event` thì so với `event_auto` (từ VAD và diarization).

In [ ]:
if not HAS_ANS:
    print("Bỏ qua: chưa có câu trả lời mức segment.")
else:
    ref_name = "người" if human else "event_auto"
    for name in models:
        m = models[name]
        pairs = []
        for sid, r in m.items():
            ref = human.get(sid, {}).get(FOCUS_QUESTION) if human else (segs.get(sid, {}).get("event_auto") if FOCUS_QUESTION == "tcontent.event" else None)
            if ref and r["answers"].get(FOCUS_QUESTION):
                pairs.append((ref, r["answers"][FOCUS_QUESTION]))
        if pairs:
            cm = pd.crosstab(pd.Series([p[0] for p in pairs], name=ref_name), pd.Series([p[1] for p in pairs], name=name))
            print(f"\n{name}: khớp {100 * sum(a == b for a, b in pairs) / len(pairs):.0f}% trên {len(pairs)} mẫu")
            display(cm)

## Xem từng segment

Audio, transcript, mốc thời gian và bảng trả lời. Ô khác nhau giữa các nguồn được tô vàng.

In [ ]:
from IPython.display import Audio, HTML, display

def disagree(sid):
    return any(len({SRC[n][sid]["answers"].get(k) for n in SRC if sid in SRC[n]} - {None}) > 1 for k, _ in QS)

def show(sid):
    s = segs[sid]
    display(HTML(f"<h4>{sid} · event_auto=<b>{s.get('event_auto')}</b> · user={s.get('user')} · bucket={s.get('bucket')}</h4>"
                 f"<p>Lời người dùng: <b>{s.get('user_text', '')}</b><br>times={s.get('times')} · next_auto={s.get('next_auto')} · "
                 f"gap={s.get('gap_after_sec_auto')} · overlap={s.get('overlap_auto')} · file_labels={s.get('file_labels')}</p>"))
    p = os.path.join(ROOT, s["audio"]) if s.get("audio") else None
    if p and os.path.exists(p):
        display(Audio(p))
    tab = pd.DataFrame({n: {k: SRC[n].get(sid, {}).get("answers", {}).get(k) for k, _ in QS} for n in SRC})
    tab.insert(0, "câu hỏi", [QTITLE[k] for k in tab.index])
    diff = tab[list(SRC)].nunique(axis=1, dropna=True) > 1
    display(tab.style.apply(lambda r: ["background:#fff3b0" if diff[r.name] and c in SRC else "" for c in r.index], axis=1))
    for n in models:
        why = models[n].get(sid, {}).get("reasoning")
        if why:
            display(HTML(f"<details><summary>Suy nghĩ của {n}</summary><pre style='white-space:pre-wrap'>{why}</pre></details>"))

def pick(event=FILTER_EVENT, bucket=FILTER_BUCKET, only_disagree=ONLY_DISAGREE):
    ids = [sid for sid in segs if not SRC or any(sid in m for m in SRC.values())]
    ids = [i for i in ids if event is None or segs[i].get("event_auto") == event]
    ids = [i for i in ids if bucket is None or bucket in (segs[i].get("bucket") or [])]
    return [i for i in ids if not only_disagree or not SRC or disagree(i)]

ids = pick()
print(f"{len(ids)} segment khớp bộ lọc")
for sid in ids[:MAX_SHOW]:
    show(sid)

### Bộ lọc tương tác (cần `ipywidgets`)

In [ ]:
try:
    import ipywidgets as w
    evs = [None] + sorted({s.get("event_auto") for s in segs.values()})
    bks = [None] + sorted({b for s in segs.values() for b in (s.get("bucket") or [])})
    ev_w, bk_w, dis_w = w.Dropdown(options=evs, description="event"), w.Dropdown(options=bks, description="bucket"), w.Checkbox(value=ONLY_DISAGREE, description="chỉ bất đồng")
    sel = w.Dropdown(description="segment")
    out = w.Output()
    def refresh(*_):
        sel.options = pick(ev_w.value, bk_w.value, dis_w.value) or [None]
    def render(*_):
        out.clear_output()
        with out:
            if sel.value: show(sel.value)
    for x in (ev_w, bk_w, dis_w): x.observe(refresh, "value")
    sel.observe(render, "value")
    refresh(); display(w.HBox([ev_w, bk_w, dis_w]), sel, out)
except ImportError:
    print("Không có ipywidgets: dùng các biến FILTER_* ở đầu notebook.")